<a href="https://colab.research.google.com/github/keudithmanco-gif/Material-de-Analitica-de-Negocios-UDEA/blob/main/M4_S42_Dashboards_Interactividad__.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p><img alt="banner" height="252px" width="1080px" src="https://docs.google.com/uc?export=download&id=14dKJeV3Ry8BubJs4BBV02c69usbHzTex"  align="center" hspace="10px" vspace="0px" ></p>

# <font color='056938'> **1. Introducción** </font>

En la [sección anterior](https://colab.research.google.com/drive/1Nn4vx9V3X6wqaSG7-YdjtZATWy5nsH7c?usp=sharing) diseñamos el `Layout` de un tablero de datos usando `Dash`. En esta sección abordaremos la tarea de hacer que el tablero sea interactivo respondendo a las acciones del usuario.







# <font color='056938'> **2. Recapitulando: datos y figuras** </font>

Necesitaremos hacer uso de los datos del precio de productos agricolas y la función para crear gráficos de linea. Esta información la leeremos desde el repositorio del curso en google drive

## <font color='8EC044'> **Leer datos desde google drive** </font>

Dado que la lectura de datos puede tomar un tiempo considerable, hemos dispuesto una base de datos preprocesada que puede leerse directamente desde google drive



In [ ]:
import pandas as pd


!gdown 1YKS6g80aELebRd3SNwCwpYVyzedtCasD
!gdown 1w2awQieBE84Qoo1iA_di3CRKBcHF2AKm

#  Leer Datos de cantidades recogidas y procesar tipos de fechas
df_precioProm = pd.read_csv('precioProm.csv', index_col=0)
df_promRec = pd.read_csv('recProm.csv')

La información cargada ha sido almacenada en el dataframe `df_precioProm` que contiene los precios históricos por producto en cada plaza de mercado





Haremos un pequeño procesamiento de las columnas ajustando el tipo de datos y creando columnas especificas para el año y mes con el fin de facilitar el filtro de los datos posteriormente

In [ ]:
# Seleccionamos las columnas necesarias
df_precioProm = df_precioProm[['ciudad', 'codProducto', 'fechaCaptura',
       'precioPromedio', 'producto']]

df_precioProm['fechaCaptura'] = pd.to_datetime(df_precioProm['fechaCaptura'])

# Creamos columnas año y mes
df_precioProm['año'] = df_precioProm['fechaCaptura'].dt.year
df_precioProm['mes'] = df_precioProm['fechaCaptura'].dt.month

# reset index and order by column
df_precioProm.sort_values(by='fechaCaptura', inplace=True)
df_precioProm.reset_index(drop=True,inplace=True)


Esta es una muestra de la información leida:

In [ ]:
df_precioProm.head(5)

## <font color='8EC044'> **Funciones que retornan gráficos** </font>

Para la creación de tableros es de utilidad definir funciones que dados unos argumentos construyan y retornen un gráfico.

En nuestro caso definimos una función que creará un gráfico para visualizar la evolución del precio de un producto en una ciudad específica



### <font color='157699'> **Gráfico de lineas** </font>

La función `grafico_precios() ` recibirá el DataFrame `df_PrecioProm`, el nombre de un producto (`producto`) y una ciudad (`ciudad`) y retornara una figura de plotly con el gráfico de lineas.

In [ ]:
import plotly.express as px

def grafico_precios(df, producto, ciudad):
  # filtra el dataframe
  df_filtered = df[(df['producto']==producto) & (df['ciudad']==ciudad)][['fechaCaptura', 'precioPromedio']]

  # Sort the values based on recording data
  df_filtered.sort_values(by=['fechaCaptura'], inplace=True)

  # create the figure
  fig = px.line(df_filtered, x='fechaCaptura', y='precioPromedio',
                title="Precio por kg de {} en las plaza de mercado de {}".format(producto, ciudad),
                labels={'precioPromedio': 'precio (kg)',
                        'fechaCaptura': 'Fecha registro'}
                )

  fig.update_layout(title_x=0.5)
  return fig



In [ ]:
producto = "Ahuyama"
ciudad = "BARRANQUILLA"

figure = grafico_precios(df_precioProm, producto, ciudad)
figure.show()

# <font color='056938'> **3. Dash Plotly** </font>

Dash es un *framework* creado por `Plotly` para crear aplicaciones web interactivas. Dentro de sus principales carácteristicas estan que:



* Las  aplicaciones se despliegan en el navegador web
* Las aplicaciones pueden crearse usando solamente `python`
* Esta diseñada con un enfoque *low-code* de modo que con pocas lineas de código pueden generarse aplicaciones completas
* Es de código abierto
* Cuenta con una [documentación](https://dash.plotly.com/) amplia y clara

Las aplicaciones generadas en `Dash` se componen de dos bloque principalmente:
> El `layout` describe como luce y se experiementa la aplicación. Considera elementos como titulo, texto, graficos menu desplegables. Para estos elementos define su ubicación, tamaño, color y demas propiedades. Para ese fin `Dash` contiene  un componente de html (**Dash HTML components**) que permite crear y dar estilo a diverso contenido, tal como, encabezados, parrafos e imagenes. Elementos como gráficas, menús, botones y deslizadores son creados usando el componente central de la aplicación (**Dash Core components**)

> La interactividad que se define a traves de `Callbacks`, las cuales son funciones que al usuario interactuar con las aplicaciones. Por ejemplo, son estas funciones las que definen la actividad que se desencadena al dar clic en un botón o un menú desplegable.

En esta sección nos concentraemos en programar la interactividad de la aplicación


## <font color='8EC044'> **Instalación** </font>

Usualmente los dos componentes principales de `Dash` se instalan con las siguientes lineas de código:

```python
import dash
from dash import html
from dash import dcc
```

<!-- Sin embargo, dado que en el caso de este tutorial deseamos correr la aplicación desde el notebook, requerimos instalar una versión especificamente diseñada para ello

```python
!pip install -q jupyter-dash
``` -->

In [ ]:
!pip install dash==3.2.0
!pip install -q dash-bootstrap-components

from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px
import numpy as np
import pandas as pd

## <font color='8EC044'> **Interactividad de la aplicación** </font>

La interacción entre los elementos del layout y las acciones del usuario se realizan a través de funciones de  que son llamadas por `Dash` cuando las propiedades de entrada de uno de los componentes cambia. Una función `callback`  se utiliza para actualizar el contenido de una aplicación web en función de las interacciones del usuario o los cambios en el estado de la aplicación (ej. al dar clic en un boton o ingresar una valor en un cuadro de texto)





### <font color='157699'> **`Callbacks` básicos** </font>

La estructura básica de una función `callback` es la siguiente

```python
@app.callback(
    Output(component_id='my_output', component_property='property'),
    Input(component_id='my_input', component_property='value')
    )
def my_function(input_value):
    # Funcionalidad para actualizar component
    return return_value
```


Las entradas (`inputs`) y salidas (`outputs`) de la aplicación se describen como argumentos de la función decoradora (`decorator`) `callback`
 >  Los decoradores son funciones que envuelven otras funciones o métodos, lo que te permite extender o modificar su funcionalidad.

Los decoradores se denotan con el símbolo `@` seguido del nombre de la función decoradora (en esta caso `callback`). Se aplican a la función o método inmediatamente antes de su definición. Cuando se llama a la función decorada, la función decoradora también se ejecuta

En esta función `callback` se identifica el componente que se desea modificar como la salida (`Output`) a través de su `id` y la propiedad que se afectará. En el siguiente ejemplo, el componente con `id`  igual a `my_output` y su propiedad `property`:

```python
Output(component_id='my_output', component_property='property')
```
De forma similar, la función `callback` identifica a través del `id` el componente de entrada (`Input`) que al ver modificada una de sus propiedades desencadena la ejecución de la función decorada. En el siguiente ejemplo, un cambio en la propiedad `value` del componente con `id` igual a `'my_input'` desencadenará la ejecución de la función decorada `my_function`

```python
Input(component_id='my_input', component_property='value')
```

La función `my_function` recibe como argumento la propiedad del componente de entrada `'my_input'` que fue modificada `'value'` y ejecuta la funcionalidad codificada para retornar el nuevo valor de la propiedad `'property'` en el componente de salida `my_output`




El siguiente ejemplo Creamos una aplicación que tiene un menú desplegable (`Dropdown`) con nombres de colores y un botón cuyo color cambia a medida que escojamos un color diferente en el menú.

> Note que hemos agregado la propiedad `id` a cada elemento para que nos sea fácil reconocerlos

La función `callback` tiene como salida (`Output`) la propiedad style del elemento con `id='colorful-button'`. Es decir, el botón que agregamos al layout. De manera similar, la función `callback` tiene como entrada (`Input`) la propiedad value del elementi con `id='color-dropdown'`. Es decir el menú desplegable que creamos en el layout

Cada vez que la propiedad `value` del menú desplegable cambie, lo que sucede cada que escogemos un color diferente, se invoca la función `update_button_color()` pasando cómo argumento el color que escogimos. El código dentro de esta función genera el nuevo valor que tendra la propiedad `style` del botón



In [ ]:
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output, State

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Change the color of the button:"),
    dcc.Dropdown(
        id='color-dropdown',
        options=['red', 'blue', 'green', 'yellow'],
        value='red'
    ),
    html.Br(),
    html.Button("Color", id='colorful-button', style={'float': 'right'})
])

@app.callback(
    Output('colorful-button', 'style'),
    Input('color-dropdown', 'value')
)
def update_button_color(selected_color):
    button_color = {'backgroundColor': selected_color}
    return button_color

# Run the app
#app.run(jupyter_mode="external")
app.run(jupyter_mode="external")

#### <font color='46B8A9'> **Ejercicio** </font>

Considere la siguiente aplicación en la que hemos dispuesto un botón (`dbc.Button`) y un área de texto (`dcc.Textarea`).

Complete  la función `callback` que permita que  cada vez que se de clic en el boton, se generé un número aleatorio entre 1 y 10 que se despliega en un area de texto

In [ ]:
import random
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output, State

app = dash.Dash(__name__,  external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row(html.H1("Random number generator")),
    dbc.Row([
        dbc.Col(html.P("Generate Random Number"), width=4),
        dbc.Col(dcc.Textarea(id='random-number-output', value="", readOnly=True), width=2)
        ]),
    dbc.Row(
          dbc.Col(dbc.Button("Generar", id='random-button'),
                  width={"size": 1, "offset": 5})
          )
    ]
  )

@app.callback(
    Output('random-number-output', 'value'),
    Input('random-button', 'n_clicks')
)
def generate_random_number(n_clicks):
    if n_clicks is None:
        return ""

    random_number = random.randint(1, 10)
    return random_number

# Run the app
app.run(jupyter_mode="external")


### <font color='157699'> **Multiples salidas (`Output`) y/o entradas (`Input`)** </font>

Es posible considerar múltiples (`Output`) y/o entradas (`Input`). Es importante considerar que:
* La función debe retornar tantos elementos como `Output` tenga la función callback y en el orden en que los `Output` se declararon
* La función a ejecutar debe recibir por lo menos tantos `Input` como los declarados en la función callback y estos se pasarán en el orden en el que se definen los `Input`

En el siguiente ejemplo, creamos una aplicación que:
* Retorna una figura  como salida (`Output`), en este caso un histograma: `dcc.Graph(id='pdf')`
* Recibe dos entradas (`Input`), en este caso el valor `value` de dos elementos Input:
  * El tamaño de la muestra de una distribución normal: `dcc.Input(id='n_sample', ...)`
  * El número de intervalos (`bins`) del histograma: `dcc.Input(id='n_bins', ...)`

In [ ]:
import dash
from dash import html
from dash import dcc
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
import numpy as np
import plotly.express as px

data = np.random.normal(0, 5, 100)
pdf = px.histogram(data, nbins=10)

app = dash.Dash(__name__,  external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row(html.H1("Histogram with k bins")),
    dbc.Row([
        dbc.Col(html.P("Sample size"), width=4),
        dbc.Col(dcc.Input(id='n_sample', type='number', value=100, min=10), width=2),
        dbc.Col(html.P("number of bins"), width=4),
        dbc.Col(dcc.Input(id='n_bins', type='number', value=6, min=2), width=2),
        ]),
    dbc.Row(dcc.Graph(id='pdf'))
    ]
  )

@app.callback(
    Output('pdf', 'figure'),
    Input('n_sample', 'value'),
    Input('n_bins', 'value')
)
def generate_random_number(n_sample, n_bins):
  # Generate a random sample of data
  np.random.seed(0)
  data = np.random.normal(loc=0, scale=1, size=1000)  # Mean = 0, Std = 1, 1000 samples
  # Create histogram
  fig = px.histogram(data, nbins=30, title='Histogram of Fake Data')

  data = np.random.normal(0, 5, n_sample)  # Mean=0, Standard Deviation=5, n_sample data points
  # # Create the graph
  fig = px.histogram(data, nbins=n_bins, title=f'Histogram with {n_bins} Bins')
  fig.update_layout(
    xaxis_title='Value',
    yaxis_title='Count',
    template='plotly_white'
  )

  return fig

# Run the app
#app.run(jupyter_mode="external")
app.run(debug=True, jupyter_mode="external")

### <font color='157699'> **Argumento `State` en callbacks** </font>

En algunas ocasiones  se necesita el valor actual de la propiedad de algun componente, es decir, el estado (`State`) para ser usado por la función que generá la salida (`Output`), para lo cual se pasa como argumento al callback dentro de la propiedad estado (`State`).

A diferencia de las entradas (`Input`), los estados (`State`) no desencadenan la función `callback`.

Consideremos el ejercicio anterior, en el que generamos un número aleatorio cada vez que damos clic en el boton `Generar`. Además de mostrar el número generado deseamos visualizar la suma acumulada de todos los números que se hayan generado. Para ello, disponemos una nueva area de texto (dcc.Textarea) con `id='sum-state'`. Requerimos pasarle a la función `callback` el valor actual en el área de texto `sum-state` para que pueda calcular el acumulado. Dado que solo requerimos el estado y no que esto desencadene la función `callback` usamos la propiedad `State`


In [ ]:
import dash
from dash import html, dcc
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output, State
import random

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row(
        html.H1("Generador de números aleatorios")
        ),

    dbc.Row(
        [dbc.Col(html.P("Número generado"),width=4),
        dbc.Col(dcc.Textarea(id="random-number-output",value="",readOnly=True),width=2)
        ]
        ),

    dbc.Row(
        [dbc.Col(html.P("Suma acumulada"),width=4),
        dbc.Col(dcc.Textarea(id="sum-state",value="0",readOnly=True),width=2)
        ]
        ),

    dbc.Row(dbc.Col(dbc.Button("Generar",id="random-button",n_clicks=0),width={"size": 1, "offset": 5})
          )
    ]
  )

@app.callback(
    Output("random-number-output", "value"),
    Output("sum-state", "value"),
    Input("random-button", "n_clicks"),
    State("sum-state", "value")
)
def generate_random_number(n_clicks, accumulated):
    # Antes de presionar el botón
    if not n_clicks:
        return "", "0"

    # Genera un número entero entre 1 y 10
    random_number = random.randint(1, 10)

    # Convierte el acumulado a entero y suma el nuevo número
    total = int(accumulated or "0") + random_number

    # Convierte los resultados a texto para mostrarlos
    return str(random_number), str(total)


# Ejecuta la aplicación
app.run(debug=True, jupyter_mode="external")

### <font color='157699'> **Callbacks encadenados** </font>

También se pueden encadenar salidas y entradas: la salida de una función `callback` podría ser la entrada de otra función `callback`.

Este patrón puede utilizarse para crear interfaces de usuario dinámicas en las que, por ejemplo, un componente de entrada actualiza las opciones disponibles de otro componente de entrada.

En el siguiente ejemplo el primer `callback` actualiza las opciones disponibles en el segundo componente `dcc.RadioItems` (identificado con el `id='cities-radio'`) basándose en el valor seleccionado en el primer componente dcc.RadioItems (identificado con el `id='department-radio'`)

El segundo `callback` establece un valor inicial en el segundo componente `dcc.RadioItems` (identificado con el `id='cities-radio'`)cuando la propiedad `options` cambia: lo establece en el primer valor de la lista de opciones.

el último `callback` usa el valor seleccionado de cada componente para actualizar el mensaje que se muestra. Si cambia el valor del primer componente `dcc.RadioItems` (identificado con el `id='department-radio'`), Dash esperará hasta que se actualice el valor del segundo `dcc.RadioItems` componente (identificado con el `id='cities-radio'`) antes de llamar al callback final. Esto previene que los callbacks sean llamados con estados inconsistentes como con "Antioquia" y "Cali".

In [ ]:
from dash.dependencies import Input, Output, State


app = dash.Dash(__name__)

# Dictionary of options
all_options = {
    'Antioquia': ['Medellín', 'Envigado', 'Turbo', 'Caucasia'],
    'Valle': ['Cali', 'Palmira', 'Tulua']
}
app.layout = html.Div([
    dcc.RadioItems(
        options = list(all_options.keys()),
        value = 'Antioquia',
        id='department-radio',
    ),

    html.Hr(),

    dcc.RadioItems(id='cities-radio'),

    html.Hr(),

    html.Div(id='display-selected-values')
])


@app.callback(
    Output('cities-radio', 'options'),
    Input('department-radio', 'value'))
def set_cities_options(selected_country):
    return [{'label': i, 'value': i} for i in all_options[selected_country]]


@app.callback(
    Output('cities-radio', 'value'),
    Input('cities-radio', 'options'))
def set_cities_value(available_options):
    return available_options[0]['value']


@app.callback(
    Output('display-selected-values', 'children'),
    Input('department-radio', 'value'),
    Input('cities-radio', 'value'))
def set_display_children(selected_country, selected_city):
    return f'{selected_city} esta ubicado en {selected_country}'

# Run the app
app.run(jupyter_mode="external")


## <font color='46B8A9'> **Ejercicio** </font>

Considere el tablero que se muestra en el siguiente video.

[![pic](https://drive.google.com/uc?id=19cfZl1jB6UqEiGwgKVcjqI-ACvPAyLO0)](https://drive.google.com/file/d/1KUiWo6WEHcXe4N9XeuVqejAmirfpqUdG/view?usp=sharing)








Complete el siguiente codigo para crear el tablero interactivo

In [ ]:
import pandas as pd
import dash
from dash import html
from dash import dcc
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
import random


!gdown 1YKS6g80aELebRd3SNwCwpYVyzedtCasD
!gdown 1w2awQieBE84Qoo1iA_di3CRKBcHF2AKm

#  Leer Datos de cantidades recogidas y procesar tipos de fechas
df_precioProm = pd.read_csv('precioProm.csv', index_col=0)

# Create dash app
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Create static graph
cities = list(df_precioProm['ciudad'].unique())
cities.sort()

# Define Layout
app.layout = dbc.Container([
      dbc.Row(html.Img(src='https://raw.githubusercontent.com/endorgobio/IntroduccionAnaliticaPrescriptiva/master/src/visualizacion.png', style={'width': '100%'})),
      dbc.Row(html.Br()),
      dbc.Row([
          dbc.Col(dcc.Dropdown(
                id='cities-dropdown',
                options=cities,
                value=cities[0]
              ), width=6),
          dbc.Col(dcc.Dropdown(id='pdts-dropdown'), width=6),
          ]),
      dbc.Row(dcc.Graph(id='prices-graph')),
      dbc.Row(html.Br()),
      dbc.Row(
          dbc.Col(dbc.Button("Guardar", id='export_grap'),
                  width={"size": 1, "offset": 11}),
          justify="end"
          ),
      html.Div(id='display-selected-values')
      ],
      fluid=True
    )

@app.callback(
    Output('pdts-dropdown', 'options'),
    Input('cities-dropdown', 'value'))
def set_product_options(selected_city):
  df_filtered = df_precioProm[df_precioProm['ciudad']==selected_city]
  products = list(df_filtered['producto'].unique())
  products.sort()
  return products

@app.callback(
    Output('pdts-dropdown', 'value'),
    Input('pdts-dropdown', 'options'))
def set_product_value(available_options):
    return available_options[0]


@app.callback(
    Output('prices-graph', 'figure'),
    Input('cities-dropdown', 'value'),
    Input('pdts-dropdown', 'value'))
def set_display_children(city, product):
  figure = grafico_precios(df_precioProm, product, city)

  return figure



# Run the app
app.run(jupyter_mode="external")

In [ ]:
df_precioProm

In [ ]:
## <font color='46B8A9'> **Ejemplo** </font>

In [ ]:
import dash
import dash_bootstrap_components as dbc
from dash import Input, Output, dcc, html

# the style arguments for the sidebar. We use position:fixed and a fixed width
SIDEBAR_STYLE = {
  "position": "fixed",
  "top": 0,
  "left": 0,
  "bottom": 0,
  "width": "20rem",
  "padding": "2rem 1rem",
  "background-color": "#f8f9fa",
}

# the styles for the main content position it to the right of the sidebar and
# add some padding.
CONTENT_STYLE = {
  "margin-left": "20rem",
  "margin-right": "2rem",
  "padding": "2rem 1rem",
}

sidebar = html.Div(
  [
    html.H4("Variación de precios agrícolas", className="display-4"),
    html.Hr(),
    html.P("Los precios de los productos agrícolas en Colombia dependen de las condiciones del mercado, la negociación y la región.", className="lead"),
    html.Img(src='https://raw.githubusercontent.com/endorgobio/IntroduccionAnaliticaPrescriptiva/master/src/visualizacion.png', style={'width': '100%'}),
    dbc.Nav(
      [
        dbc.NavLink("Problemática", href="/", active="exact"),
        dbc.NavLink("Análisis por producto", href="/page-1", active="exact"),
        dbc.NavLink("Analisis por ciudades", href="/page-2", active="exact"),
      ],
      vertical=True,
      pills=True,
    ),
  ],
  style=SIDEBAR_STYLE,
)

content = html.Div(id="page-content", style=CONTENT_STYLE)

# Create app
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Define Layout
app.layout = html.Div([dcc.Location(id="url"), sidebar, content])

@app.callback(
  Output("page-content", "children"),
  [Input("url", "pathname")])
def render_page_content(pathname):
  if pathname == "/":
    return html.P("Este es el contenido de la página principal!")
  elif pathname == "/page-1":
    return html.P("Este es el contenido de la página de productos")
  elif pathname == "/page-2":
    #return html.P("Este es el contenido de la página ciudades")
    return dbc.Card(
    dbc.CardBody(
        [
            html.H4("Ventas Totales", className="card-title"),
            html.P("$492m", className="card-text", style={"fontSize": 30}),
            html.P("▲ 12% respecto al mes anterior", className="card-text text-success"),
        ]
    ),
    style={"width": "18rem", "margin": "10px"},
)
# If the user tries to reach a different page, return a 404 message
  return html.Div(
  [
    html.H1("404: Not found", className="text-danger"),
    html.Hr(),
    html.P(f"The pathname {pathname} was not recognised..."),
  ],
  className="p-3 bg-light rounded-3",
)

app.run(jupyter_mode="external")


